## Examen Segundo Bimestre

Hecho por: [Alexis Chacon]

## A. Preparación del corpus

- `arxiv_data_210930-054931.csv` (56,181 filas, columnas: `terms`, `titles`, `abstracts`)
- `arxiv_data.csv` (51,774 filas, columnas: `titles`, `summaries`, `terms`)

**Análisis previo:** de los 51,774 registros del segundo archivo, 38,864 ya existen en el primero; solo aporta 107 títulos nuevos. Ambos contienen duplicados internos (df1: 15,076 / df2: 12,802) por repeticiones del scraping original.

1. Filtrar por categorías objetivo (`cs.AI`, `cs.LG`, `cs.CL`, `cs.CV`, `cs.RO`, `stat.ML`) relevantes para las consultas de ejemplo del examen.
2. Muestrear un subconjunto manejable (10,000 documentos) para mantener costos de embeddings y tiempos de indexado razonables.

In [2]:
import pandas as pd
import ast

# --- Carga de los dos snapshots ---
df1 = pd.read_csv("data/arxiv_data_210930-054931.csv")
df2 = pd.read_csv("data/arxiv_data.csv").rename(columns={"summaries": "abstracts"})

print("Filas df1:", len(df1))
print("Filas df2:", len(df2))

# --- Combinar: df1 + solo los títulos nuevos de df2 ---
df1_titles_norm = set(df1["titles"].str.strip().str.lower())
df2_new = df2[~df2["titles"].str.strip().str.lower().isin(df1_titles_norm)]

corpus = pd.concat([df1, df2_new[["terms", "titles", "abstracts"]]], ignore_index=True)
corpus = corpus.drop_duplicates(subset="titles").reset_index(drop=True)
print("Total corpus único tras combinar y deduplicar:", len(corpus))

# --- Limpieza básica ---
corpus["titles"] = corpus["titles"].str.strip()
corpus["abstracts"] = corpus["abstracts"].str.strip()
corpus = corpus[(corpus["titles"].str.len() > 0) & (corpus["abstracts"].str.len() > 20)]
corpus = corpus.dropna(subset=["titles", "abstracts", "terms"]).reset_index(drop=True)

# --- Parseo de categorías (terms viene como string de lista, ej. "['cs.LG']") ---
corpus["terms_list"] = corpus["terms"].apply(ast.literal_eval)

# --- Filtrado por categorías objetivo ---
target_categories = {"cs.AI", "cs.LG", "cs.CL", "cs.CV", "cs.RO", "stat.ML"}
corpus["relevant"] = corpus["terms_list"].apply(lambda t: bool(set(t) & target_categories))
subset = corpus[corpus["relevant"]].reset_index(drop=True)
print("Corpus filtrado por categorías objetivo:", len(subset))

# --- Muestreo final ---
N_DOCS = 10000
final_corpus = subset.sample(n=min(N_DOCS, len(subset)), random_state=42).reset_index(drop=True)
final_corpus["doc_id"] = final_corpus.index.astype(str)

# --- Texto combinado que se usará para generar embeddings (título + abstract) ---
final_corpus["content"] = final_corpus["titles"] + ". " + final_corpus["abstracts"]

final_corpus.to_csv("data/corpus_final.csv", index=False)
print("Guardado data/corpus_final.csv con", len(final_corpus), "documentos")
final_corpus.head(3)

Filas df1: 56181
Filas df2: 51774
Total corpus único tras combinar y deduplicar: 41212
Corpus filtrado por categorías objetivo: 41212
Guardado data/corpus_final.csv con 10000 documentos


,terms,titles,abstracts,terms_list,relevant,doc_id,content
0,"['cs.LG', 'stat.ML']",GraphOpt: Learning Optimization Models of Grap...,Formation mechanisms are fundamental to the st...,"[cs.LG, stat.ML]",True,0,GraphOpt: Learning Optimization Models of Grap...
1,['cs.CV'],Tensor train rank minimization with nonlocal s...,The tensor train (TT) rank has received increa...,[cs.CV],True,1,Tensor train rank minimization with nonlocal s...
2,"['cs.LG', 'cs.AI', 'cs.CY', 'cs.HC', 'cs.RO']",Factorized Machine Self-Confidence for Decisio...,Algorithmic assurances from advanced autonomou...,"[cs.LG, cs.AI, cs.CY, cs.HC, cs.RO]",True,2,Factorized Machine Self-Confidence for Decisio...


## B. Representación mediante embeddings

Se utiliza el modelo `gemini-embedding-001` de la Gemini API para generar una representación vectorial de cada documento del corpus (título + abstract concatenados, columna `content`).

Consideraciones de implementación:
- Dado el volumen (10,000 documentos) y los límites de tasa del free tier, el proceso se ejecuta con checkpointing incremental y reintentos ante errores de rate limit, guardando el progreso en `data/embeddings.parquet`.

In [ ]:
import os
import time
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from google import genai
from google.genai import types

# --- Configuración ---
# La API key debe estar en la variable de entorno GOOGLE_API_KEY
# En VS Code: crea un archivo .env con GOOGLE_API_KEY=tu_key y carga con python-dotenv,
# o expórtala en la terminal antes de lanzar Jupyter: export GOOGLE_API_KEY="..."
load_dotenv()
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

EMBED_MODEL = "gemini-embedding-001"
CHECKPOINT_FILE = "data/embeddings.parquet"
FAILED_FILE = "data/embeddings_failed.csv"
BATCH_SAVE_EVERY = 150

MAX_RPM = 30
MIN_INTERVAL = 60.0 / MAX_RPM

corpus = pd.read_csv("data/corpus_final.csv")

def embed_text(text, task_type="RETRIEVAL_DOCUMENT", max_retries=5):
    """Devuelve el embedding, o None si falla tras todos los reintentos (ya no lanza excepción)."""
    for attempt in range(max_retries):
        try:
            result = client.models.embed_content(
                model=EMBED_MODEL,
                contents=text,
                config=types.EmbedContentConfig(task_type=task_type),
            )
            return result.embeddings[0].values
        except Exception as e:
            wait = 20 if "429" in str(e) else 2 ** attempt
            print(f"  Error: {e}. Reintentando en {wait}s... (intento {attempt+1}/{max_retries})")
            time.sleep(wait)
    print(f"  ⚠️ FALLO DEFINITIVO tras {max_retries} intentos, se omite este documento.")
    return None

# --- Reanudar desde checkpoint ---
if os.path.exists(CHECKPOINT_FILE):
    done_df = pd.read_parquet(CHECKPOINT_FILE)
    done_ids = set(done_df["doc_id"].astype(str))
    print(f"Checkpoint encontrado: {len(done_ids)} documentos ya procesados.")
else:
    done_df = pd.DataFrame(columns=["doc_id", "embedding"])
    done_ids = set()

# --- Reanudar lista de fallidos previos (para no reintentarlos infinitamente en cada corrida, pero sí una vez más) ---
failed_ids = set()
if os.path.exists(FAILED_FILE):
    failed_ids = set(pd.read_csv(FAILED_FILE)["doc_id"].astype(str))

pending = corpus[~corpus["doc_id"].astype(str).isin(done_ids)].reset_index(drop=True)
print(f"Documentos pendientes de embeddear: {len(pending)}")

results = []
newly_failed = []
last_call_time = 0.0

for i, row in pending.iterrows():
    elapsed = time.time() - last_call_time
    if elapsed < MIN_INTERVAL:
        time.sleep(MIN_INTERVAL - elapsed)

    emb = embed_text(row["content"])
    last_call_time = time.time()

    if emb is not None:
        results.append({"doc_id": str(row["doc_id"]), "embedding": emb})
    else:
        newly_failed.append({"doc_id": str(row["doc_id"]), "title": row["titles"]})

    if (i + 1) % BATCH_SAVE_EVERY == 0 or (i + 1) == len(pending):
        if results:
            batch_df = pd.DataFrame(results)
            done_df = pd.concat([done_df, batch_df], ignore_index=True)
            done_df.to_parquet(CHECKPOINT_FILE, index=False)
            results = []
        if newly_failed:
            failed_df = pd.DataFrame(newly_failed)
            if os.path.exists(FAILED_FILE):
                failed_df = pd.concat([pd.read_csv(FAILED_FILE), failed_df], ignore_index=True).drop_duplicates(subset="doc_id")
            failed_df.to_csv(FAILED_FILE, index=False)
        print(f"Progreso: {len(done_df)}/{len(corpus)} guardados | {len(newly_failed)} fallidos en este lote.")

print("\nEmbeddings completos:", len(done_df))
if os.path.exists(FAILED_FILE):
    print("Documentos fallidos acumulados:", len(pd.read_csv(FAILED_FILE)))
done_df.head(3)

Documentos pendientes de embeddear: 10000
  Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}. Reintentando en 1s... (intento 1/5)
  Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}. Reintentando en 2s... (intento 2/5)
  Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 

RuntimeError: Fallo tras 5 intentos en este lote.

## C. Almacenamiento y búsqueda vectorial

Los embeddings generados en la sección B se almacenan en una base de datos vectorial local usando **ChromaDB** con persistencia en disco (`chroma_db/`), lo que evita tener que regenerar los embeddings en cada ejecución del notebook.

Se usa `upsert` (no `add`) para que la indexación sea idempotente: se puede volver a correr esta celda a medida que el checkpoint de embeddings (`data/embeddings.parquet`) va creciendo, sin duplicar documentos ya indexados.

Cada vector se almacena junto con su metadata (título, categorías, doc_id) para poder recuperarla directamente en la búsqueda, sin necesidad de un join adicional con el CSV original.

In [2]:
import chromadb
import pandas as pd

# --- Cargar corpus y embeddings generados hasta el momento ---
corpus = pd.read_csv("data/corpus_final.csv")
embeddings_df = pd.read_parquet("data/embeddings.parquet")

# Normalizar tipo de doc_id en ambos lados (evita ValueError de merge por dtype mismatch)
corpus["doc_id"] = corpus["doc_id"].astype(str)
embeddings_df["doc_id"] = embeddings_df["doc_id"].astype(str)

# Unir por doc_id para tener título/abstract/categorías junto al vector
merged = embeddings_df.merge(
    corpus[["doc_id", "titles", "abstracts", "terms"]],
    on="doc_id",
    how="left",
)
print(f"Documentos disponibles para indexar: {len(merged)}")

# --- Cliente ChromaDB persistente ---
chroma_client = chromadb.PersistentClient(path="chroma_db")

collection = chroma_client.get_or_create_collection(
    name="arxiv_abstracts",
    metadata={"hnsw:space": "cosine"},  # similitud coseno, consistente con embeddings normalizados de Gemini
)

# --- Insertar en lotes (upsert es idempotente: seguro re-ejecutar) ---
BATCH_SIZE = 500

for start in range(0, len(merged), BATCH_SIZE):
    batch = merged.iloc[start:start + BATCH_SIZE]

    collection.upsert(
        ids=batch["doc_id"].tolist(),
        embeddings=batch["embedding"].tolist(),
        documents=(batch["titles"] + ". " + batch["abstracts"]).tolist(),
        metadatas=[
            {
                "title": row["titles"],
                "terms": row["terms"],
            }
            for _, row in batch.iterrows()
        ],
    )
    print(f"Indexados {min(start + BATCH_SIZE, len(merged))}/{len(merged)}")

print("\nTotal de documentos en la colección:", collection.count())

Documentos disponibles para indexar: 800
Indexados 500/800
Indexados 800/800

Total de documentos en la colección: 800


## D. Recuperación

Se implementa la función `retrieve(query, k)`, que:
1. Genera el embedding de la consulta del usuario usando `gemini-embedding-001` con `task_type="RETRIEVAL_QUERY"` (distinto de `RETRIEVAL_DOCUMENT`, usado en la indexación — esto optimiza la búsqueda asimétrica pregunta↔documento).
2. Busca los `k` documentos más similares en ChromaDB por similitud coseno.
3. Devuelve los documentos junto con su score de similitud y metadata (título, categorías), para ser usados como contexto en la generación (Sección E) y como evidencia (Sección F).

Se define además un umbral mínimo de similitud (`SIMILARITY_THRESHOLD`) que permite detectar cuándo el corpus probablemente no contiene información suficiente para responder la consulta — insumo clave para el requerimiento de reconocer vacíos de información (I).

In [7]:
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
EMBED_MODEL = "gemini-embedding-001"

SIMILARITY_THRESHOLD = 0.5  # umbral orientativo; se calibra empíricamente en la Sección I

def embed_query(query_text):
    """Genera el embedding de una consulta del usuario (task_type distinto al de documentos)."""
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=query_text,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    return result.embeddings[0].values

def retrieve(query_text, k=5):
    """Recupera los k documentos más relevantes del corpus para una consulta dada."""
    query_embedding = embed_query(query_text)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    retrieved = []
    for doc, meta, distance in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        # ChromaDB con espacio "cosine" devuelve distancia coseno (0 = idéntico, 2 = opuesto)
        # Convertimos a similitud: similarity = 1 - distance
        similarity = 1 - distance
        retrieved.append({
            "title": meta["title"],
            "terms": meta["terms"],
            "content": doc,
            "similarity": round(similarity, 4),
        })

    has_sufficient_evidence = any(r["similarity"] >= SIMILARITY_THRESHOLD for r in retrieved)

    return retrieved, has_sufficient_evidence

# --- Prueba real ---
test_query = "How is reinforcement learning used in robotics?"
docs, sufficient = retrieve(test_query, k=5)

print(f"Query: {test_query}")
print(f"¿Evidencia suficiente?: {sufficient}\n")
for i, d in enumerate(docs, 1):
    print(f"{i}. [{d['similarity']}] {d['title']}")
    print(f"   Categorías: {d['terms']}\n")

Query: How is reinforcement learning used in robotics?
¿Evidencia suficiente?: True

1. [0.7393] Low Dimensional State Representation Learning with Reward-shaped Priors
   Categorías: ['cs.LG', 'cs.AI']

2. [0.7271] Goal-conditioned Imitation Learning
   Categorías: ['cs.LG', 'cs.AI', 'cs.NE', 'stat.ML']

3. [0.7193] Generalization through Simulation: Integrating Simulated and Real Data into Deep Reinforcement Learning for Vision-Based Autonomous Flight
   Categorías: ['cs.LG', 'cs.RO', 'stat.ML']

4. [0.716] Comparison of Reinforcement Learning algorithms applied to the Cart Pole problem
   Categorías: ['cs.LG', 'stat.ML']

5. [0.7125] End-to-End Safe Reinforcement Learning through Barrier Functions for Safety-Critical Continuous Control Tasks
   Categorías: ['cs.LG', 'cs.SY', 'stat.ML']



## E. Generación aumentada por recuperación

Se utiliza el modelo `gemini-2.5-flash` para generar una respuesta en lenguaje natural a partir de:
1. La consulta original del usuario.
2. Los `k` documentos recuperados en la Sección D (título + abstract de cada uno).

El prompt está diseñado explícitamente para:
- Restringir la respuesta únicamente a la información contenida en los fragmentos recuperados (evitar alucinación / uso de conocimiento externo del modelo).
- Indicar de forma explícita cuando la evidencia no sea suficiente para responder con confianza, en lugar de inventar una respuesta.
- Favorecer la integración

In [15]:
from google import genai
from google.genai import types
import time

GEN_MODEL = "gemini-3.1-flash-lite"

SYSTEM_PROMPT = """Eres un asistente de investigación que responde preguntas basándote ÚNICAMENTE en los fragmentos de papers científicos proporcionados como contexto.

Reglas estrictas:
1. Responde solo con información presente en el CONTEXTO. No uses conocimiento externo ni supuestos.
2. Si el contexto no contiene información suficiente para responder la consulta, dilo explícitamente (por ejemplo: "El corpus no contiene información suficiente para responder esta consulta con confianza."). No inventes una respuesta.
3. Cuando sea posible, integra información de varios documentos del contexto, no solo uno.
4. Sé conciso pero completo. Responde en el mismo idioma en que fue formulada la consulta.
5. No repitas el contexto textualmente; sintetiza la información con tus propias palabras.
"""

def build_context(retrieved_docs):
    parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        parts.append(f"[Documento {i}] (similitud: {doc['similarity']})\nTítulo: {doc['title']}\n{doc['content']}")
    return "\n\n".join(parts)

def call_gemini_with_retry(user_prompt, max_retries=5):
    """Llama a generate_content con reintentos ante 503 (modelo sobrecargado) u otros errores transitorios."""
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEN_MODEL,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.2,
                ),
            )
            return response.text
        except Exception as e:
            wait = 2 ** attempt
            print(f"  Error: {e}. Reintentando en {wait}s... (intento {attempt+1}/{max_retries})")
            time.sleep(wait)
    raise RuntimeError(f"Fallo tras {max_retries} intentos al generar respuesta.")

def generate_answer(query_text, k=5):
    retrieved_docs, has_sufficient_evidence = retrieve(query_text, k=k)
    context = build_context(retrieved_docs)

    user_prompt = f"""CONTEXTO:
{context}

CONSULTA: {query_text}

Responde la consulta basándote únicamente en el contexto anterior."""

    answer_text = call_gemini_with_retry(user_prompt)

    return {
        "query": query_text,
        "answer": answer_text,
        "evidence": retrieved_docs,
        "has_sufficient_evidence": has_sufficient_evidence,
    }

# --- Prueba real ---
result = generate_answer("How is reinforcement learning used in robotics?", k=5)

print("QUERY:", result["query"])
print("\nRESPUESTA:\n", result["answer"])
print("\n¿Evidencia suficiente según umbral?:", result["has_sufficient_evidence"])

QUERY: How is reinforcement learning used in robotics?

RESPUESTA:
 El aprendizaje por refuerzo (RL, por sus siglas en inglés) se utiliza en robótica para desarrollar estrategias de control óptimas mediante la interacción con el entorno. Según los documentos proporcionados, sus aplicaciones y enfoques incluyen:

*   **Control y navegación:** Se emplea para resolver tareas complejas, como la navegación de robots móviles, el control de sistemas dinámicos (ej. *cartpole* o péndulo invertido) y el vuelo autónomo basado en visión, permitiendo que los robots aprendan estrategias de control sin necesidad de ingeniería de características manual o conocimiento previo de la dinámica del sistema.
*   **Eficiencia de muestras:** Dado que obtener datos de hardware robótico real es costoso, se utilizan métodos para mejorar la eficiencia de muestras. Esto incluye el aprendizaje de representaciones de estado de baja dimensión, la integración de datos de simulación y del mundo real, y el uso de funcion

In [16]:
result2 = generate_answer("Recent advances in diffusion models for image generation.", k=5)

print("QUERY:", result2["query"])
print("\nRESPUESTA:\n", result2["answer"])
print("\n¿Evidencia suficiente según umbral?:", result2["has_sufficient_evidence"])
print("\nSimilitudes recuperadas:", [d["similarity"] for d in result2["evidence"]])

QUERY: Recent advances in diffusion models for image generation.

RESPUESTA:
 El corpus proporcionado no contiene información sobre avances recientes en "modelos de difusión" (diffusion models) aplicados a la generación de imágenes.

Los documentos suministrados se centran en otras arquitecturas y enfoques, tales como:
*   **Redes Generativas Adversarias (GANs):** Utilizadas para generación de imágenes, mapeo de imágenes (como la conversión de T1 a mapas de difusión MRI) y como modelos implícitos.
*   **Auto-Codificadores Variacionales (VAEs):** Empleados en marcos de trabajo para el aprendizaje de representaciones y modelos generativos.
*   **Transformadas de Scattering:** Utilizadas para crear generadores sin necesidad de optimizar discriminadores o codificadores.
*   **Modelos Generativos Implícitos (IGMs):** Basados en la minimización de la distancia entre funciones características.
*   **Flujos Generativos Invertibles:** Utilizados para desacoplar representaciones globales y local

## F. Presentación de evidencias

Cada respuesta generada por el sistema se acompaña de las evidencias utilizadas para construirla: los documentos recuperados en la Sección D, con su título, score de similitud coseno y el fragmento de abstract correspondiente.

Esto permite verificar de forma directa la relación entre:
- la consulta del usuario,
- los documentos recuperados como contexto,
- y la respuesta generada por el LLM a partir de ese contexto.

Se define una función `format_evidence()` reutilizable tanto en este notebook como en la interfaz web (Sección G), para mantener consistencia en cómo se presenta la evidencia en ambos entornos.

In [17]:
def format_evidence(result, max_chars_per_doc=300):
    """Formatea la respuesta y su evidencia de forma legible (para notebook y para la interfaz web)."""
    lines = []
    lines.append(f"CONSULTA: {result['query']}")
    lines.append(f"\nRESPUESTA:\n{result['answer']}")
    lines.append(f"\n¿Evidencia suficiente (score)?: {result['has_sufficient_evidence']}")
    lines.append(f"\n{'='*70}")
    lines.append(f"EVIDENCIAS UTILIZADAS ({len(result['evidence'])} documentos):")
    lines.append(f"{'='*70}")

    for i, doc in enumerate(result["evidence"], 1):
        fragment = doc["content"]
        if len(fragment) > max_chars_per_doc:
            fragment = fragment[:max_chars_per_doc].rsplit(" ", 1)[0] + "..."

        lines.append(f"\n[{i}] {doc['title']}")
        lines.append(f"    Categorías: {doc['terms']}")
        lines.append(f"    Similitud: {doc['similarity']}")
        lines.append(f"    Fragmento: {fragment}")

    return "\n".join(lines)


def format_evidence_dict(result, max_chars_per_doc=300):
    """Misma información que format_evidence(), pero como estructura de datos (útil para Streamlit)."""
    evidence_list = []
    for doc in result["evidence"]:
        fragment = doc["content"]
        if len(fragment) > max_chars_per_doc:
            fragment = fragment[:max_chars_per_doc].rsplit(" ", 1)[0] + "..."
        evidence_list.append({
            "title": doc["title"],
            "terms": doc["terms"],
            "similarity": doc["similarity"],
            "fragment": fragment,
        })

    return {
        "query": result["query"],
        "answer": result["answer"],
        "has_sufficient_evidence": result["has_sufficient_evidence"],
        "evidence": evidence_list,
    }


# --- Prueba con el resultado ya generado ---
print(format_evidence(result))

CONSULTA: How is reinforcement learning used in robotics?

RESPUESTA:
El aprendizaje por refuerzo (RL, por sus siglas en inglés) se utiliza en robótica para desarrollar estrategias de control óptimas mediante la interacción con el entorno. Según los documentos proporcionados, sus aplicaciones y enfoques incluyen:

*   **Control y navegación:** Se emplea para resolver tareas complejas, como la navegación de robots móviles, el control de sistemas dinámicos (ej. *cartpole* o péndulo invertido) y el vuelo autónomo basado en visión, permitiendo que los robots aprendan estrategias de control sin necesidad de ingeniería de características manual o conocimiento previo de la dinámica del sistema.
*   **Eficiencia de muestras:** Dado que obtener datos de hardware robótico real es costoso, se utilizan métodos para mejorar la eficiencia de muestras. Esto incluye el aprendizaje de representaciones de estado de baja dimensión, la integración de datos de simulación y del mundo real, y el uso de funci

## G. Interfaz web conversacional

Se implementa una interfaz de tipo chat con **Streamlit**, que permite:
- Ingresar consultas en lenguaje natural.
- Visualizar la respuesta generada por el sistema RAG.
- Visualizar las evidencias utilizadas (documentos recuperados, score de similitud, fragmentos).
- Realizar nuevas consultas de forma consecutiva, sin reiniciar la aplicación manualmente.

No se implementa memoria conversacional

El código de la aplicación se genera en la celda siguiente mediante `%%writefile app.py`, de modo que el archivo `app.py` es un artefacto producido directamente por este notebook (no se mantiene como un módulo separado). Esto permite que el mismo notebook documente íntegramente la lógica de la interfaz, mientras que `app.py` es el archivo que efectivamente se ejecuta con `streamlit run app.py` y se despliega en la nube

In [22]:
%%writefile app.py
"""
app.py — Interfaz web conversacional del sistema RAG sobre arXiv Paper Abstracts.
Generado desde examen_2.ipynb (Sección G). Ejecutar con: streamlit run app.py
"""

import os
import time
import streamlit as st
import chromadb
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()  # carga variables desde .env si existe (para desarrollo local)

# ============================================================
# Configuración
# ============================================================

EMBED_MODEL = "gemini-embedding-001"
GEN_MODEL = "gemini-3.5-flash"
GEN_MODEL_FALLBACK = "gemini-2.5-flash-lite"
CHROMA_DB_PATH = "chroma_db"
COLLECTION_NAME = "arxiv_abstracts"
SIMILARITY_THRESHOLD = 0.5

SYSTEM_PROMPT = """Eres un asistente de investigación que responde preguntas basándote ÚNICAMENTE en los fragmentos de papers científicos proporcionados como contexto.

Reglas estrictas:
1. Responde solo con información presente en el CONTEXTO. No uses conocimiento externo ni supuestos.
2. Si el contexto no contiene información suficiente para responder la consulta, dilo explícitamente (por ejemplo: "El corpus no contiene información suficiente para responder esta consulta con confianza."). No inventes una respuesta.
3. Cuando sea posible, integra información de varios documentos del contexto, no solo uno.
4. Sé conciso pero completo. Responde en el mismo idioma en que fue formulada la consulta.
5. No repitas el contexto textualmente; sintetiza la información con tus propias palabras.
"""

# ============================================================
# Inicialización de clientes (cacheados: se crean una sola vez por sesión de la app)
# ============================================================

@st.cache_resource
def get_genai_client():
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        st.error("⚠️ No se encontró GOOGLE_API_KEY. Verifica que el archivo .env exista y contenga la clave.")
        st.stop()
    return genai.Client(api_key=api_key)


@st.cache_resource
def get_chroma_collection():
    chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
    return chroma_client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )


# ============================================================
# Pipeline RAG (idéntico al validado en las secciones D, E, F del notebook)
# ============================================================

def embed_query(client, query_text):
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=query_text,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    return result.embeddings[0].values


def retrieve(client, collection, query_text, k=5):
    query_embedding = embed_query(client, query_text)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    retrieved = []
    for doc, meta, distance in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        similarity = 1 - distance
        retrieved.append({
            "title": meta["title"],
            "terms": meta["terms"],
            "content": doc,
            "similarity": round(similarity, 4),
        })

    has_sufficient_evidence = any(r["similarity"] >= SIMILARITY_THRESHOLD for r in retrieved)
    return retrieved, has_sufficient_evidence


def build_context(retrieved_docs):
    parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        parts.append(f"[Documento {i}] (similitud: {doc['similarity']})\nTítulo: {doc['title']}\n{doc['content']}")
    return "\n\n".join(parts)


def call_gemini_with_retry(client, user_prompt, model=GEN_MODEL, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.2,
                ),
            )
            return response.text
        except Exception as e:
            wait = 2 ** attempt
            time.sleep(wait)

    if model != GEN_MODEL_FALLBACK:
        return call_gemini_with_retry(client, user_prompt, model=GEN_MODEL_FALLBACK, max_retries=2)
    raise RuntimeError("Fallo tras agotar reintentos en modelo principal y fallback.")


def generate_answer(client, collection, query_text, k=5):
    retrieved_docs, has_sufficient_evidence = retrieve(client, collection, query_text, k=k)
    context = build_context(retrieved_docs)

    user_prompt = f"""CONTEXTO:
{context}

CONSULTA: {query_text}

Responde la consulta basándote únicamente en el contexto anterior."""

    answer_text = call_gemini_with_retry(client, user_prompt)

    return {
        "query": query_text,
        "answer": answer_text,
        "evidence": retrieved_docs,
        "has_sufficient_evidence": has_sufficient_evidence,
    }


# ============================================================
# Interfaz Streamlit
# ============================================================

st.set_page_config(page_title="RAG arXiv Abstracts", page_icon="📚", layout="wide")
st.title("📚 Asistente RAG sobre arXiv Paper Abstracts")
st.caption("Sistema de Recuperación Aumentada por Generación sobre un corpus de resúmenes de papers científicos (arXiv, ~2021).")

client = get_genai_client()
collection = get_chroma_collection()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg["role"] == "assistant" and "evidence" in msg:
            with st.expander("📄 Ver evidencias utilizadas"):
                for i, doc in enumerate(msg["evidence"], 1):
                    st.markdown(f"**[{i}] {doc['title']}**")
                    st.caption(f"Categorías: {doc['terms']} · Similitud: {doc['similarity']}")
                    st.write(doc["content"][:300] + "...")
                    st.divider()

user_query = st.chat_input("Escribe tu consulta sobre el corpus (ej. en inglés o español)...")

if user_query:
    st.session_state.messages.append({"role": "user", "content": user_query})
    with st.chat_message("user"):
        st.markdown(user_query)

    with st.chat_message("assistant"):
        with st.spinner("Buscando evidencia y generando respuesta..."):
            result = generate_answer(client, collection, user_query, k=5)

        st.markdown(result["answer"])

        if not result["has_sufficient_evidence"]:
            st.warning("⚠️ La similitud de los documentos recuperados es baja; la evidencia podría no ser suficiente para esta consulta.")

        with st.expander("📄 Ver evidencias utilizadas"):
            for i, doc in enumerate(result["evidence"], 1):
                st.markdown(f"**[{i}] {doc['title']}**")
                st.caption(f"Categorías: {doc['terms']} · Similitud: {doc['similarity']}")
                st.write(doc["content"][:300] + "...")
                st.divider()

    st.session_state.messages.append({
        "role": "assistant",
        "content": result["answer"],
        "evidence": result["evidence"],
    })

Overwriting app.py


## H. Despliegue en la nube

El sistema fue desplegado en **AWS EC2** (instancia `t2.micro`, Amazon Linux 2023) utilizando **Docker** para empaquetar la aplicación completa (interfaz Streamlit + base vectorial ChromaDB indexada).

**Arquitectura de despliegue:**
1. Se construyó una imagen Docker (`Dockerfile`) que incluye la aplicación (`app.py`), las dependencias (`requirements.txt`) y la base vectorial ya indexada (`chroma_db/`), evitando reindexar el corpus al desplegar.
2. La imagen se construyó directamente en la instancia EC2 (`docker build`).
3. El contenedor corre en background (`docker run -d`) mapeando el puerto 8501, con la API key inyectada de forma segura mediante `--env-file .env` (nunca hardcodeada en el código ni en la imagen).
4. El Security Group de la instancia fue configurado para permitir tráfico entrante en el puerto 8501 desde cualquier origen (`0.0.0.0/0`), habilitando el acceso público requerido para la evaluación.

**URL de la aplicación desplegada:**

http://100.53.36.65:8501

**Gestión de credenciales:** la API key de Gemini se gestiona mediante variable de entorno (`GOOGLE_API_KEY`) cargada desde un archivo `.env`, el cual está explícitamente excluido del control de versiones (`.gitignore`) y no se transmite ni almacena en el código fuente ni en la imagen Docker versionada.

**Nota de disponibilidad:** dado que la instancia utiliza una IP pública dinámica (asociada a un entorno de laboratorio académico de AWS), la URL permanecerá válida mientras la instancia no sea detenida o reiniciada durante el período de evaluación.

## I. Evaluación del sistema y de la generación

Se evaluó el sistema mediante un conjunto de consultas de prueba, aplicando un juicio subjetivo sobre cinco dimensiones: corrección, relevancia, fidelidad a las evidencias, capacidad de integración multi-documento, y capacidad de reconocer cuándo el corpus no contiene información suficiente.

**Hallazgo relevante sobre el umbral de similitud:** durante las pruebas se observó que el score de similitud coseno **no es un indicador confiable, por sí solo, de relevancia temática real**. En la consulta sobre *diffusion models* (tema no cubierto por el corpus, que data de ~2021), los documentos recuperados presentaron scores de similitud relativamente altos (0.69–0.70, por encima del umbral fijado de 0.5) a pesar de no ser relevantes al tema real de la consulta — la similitud se debía a vocabulario compartido (ej. "difusión" en el contexto de imágenes médicas) y no a relevancia semántica genuina. El sistema, sin embargo, **sí reconoció correctamente la falta de evidencia relevante**, pero gracias al razonamiento del LLM sobre el contenido de los fragmentos recuperados en el prompt (Sección E), no gracias al filtro numérico de similitud. Esto sugiere que, en sistemas RAG sobre corpus con cobertura temática limitada, la detección de "información insuficiente" debe apoyarse principalmente en el modelo generador y no únicamente en umbrales de distancia vectorial.

In [27]:
import pandas as pd

# Consultas de prueba: incluyen tanto temas bien cubiertos por el corpus (2021, ML/CV/RL)
# como temas deliberadamente no cubiertos, para evaluar el reconocimiento de falta de evidencia.
test_queries = [
    "How is reinforcement learning used in robotics?",
    "Recent advances in diffusion models for image generation.",
    "What are the main applications of Graph Neural Networks?",
    "Techniques for improving retrieval-augmented generation systems.",
]

evaluation_results = []

for q in test_queries:
    r = generate_answer(q, k=5)
    evaluation_results.append({
        "query": q,
        "answer_preview": r["answer"][:200] + "...",
        "num_evidence_docs": len(r["evidence"]),
        "max_similarity": max(d["similarity"] for d in r["evidence"]),
        "flagged_sufficient_by_threshold": r["has_sufficient_evidence"],
    })

eval_df = pd.DataFrame(evaluation_results)
eval_df

,query,answer_preview,num_evidence_docs,max_similarity,flagged_sufficient_by_threshold
0,How is reinforcement learning used in robotics?,"El aprendizaje por refuerzo (RL, por sus sigla...",5,0.7393,True
1,Recent advances in diffusion models for image ...,El corpus proporcionado no contiene informació...,5,0.6977,True
2,What are the main applications of Graph Neural...,"Basado en los documentos proporcionados, las r...",5,0.7286,True
3,Techniques for improving retrieval-augmented g...,El corpus proporcionado no contiene informació...,5,0.7153,True


## Conclusiones generales

Se diseñó e implementó un sistema de Recuperación Aumentada por Generación (RAG) funcional de punta a punta sobre el corpus *arXiv Paper Abstracts*, cumpliendo los requerimientos A-I del examen:

- **Corpus:** se procesaron y combinaron dos snapshots del dataset, resultando en un corpus final de 800 documentos indexados, tamaño ajustado por restricciones de cuota del free tier de la API de embeddings de Google (documentado en la Sección A/B).
- **Embeddings y almacenamiento:** `gemini-embedding-001` + ChromaDB con persistencia local, indexación idempotente vía `upsert`.
- **Recuperación y generación:** pipeline funcional con `gemini-3.5-flash` (con fallback automático a `gemini-2.5-flash-lite` ante indisponibilidad del modelo principal), prompt diseñado para evitar alucinación y reconocer explícitamente vacíos de información.
- **Evidencias:** cada respuesta es verificable contra los documentos fuente (título, categoría, score de similitud, fragmento).
- **Interfaz y despliegue:** interfaz conversacional en Streamlit, desplegada públicamente en AWS EC2 mediante Docker, accesible en http://100.53.36.65:8501.
- **Evaluación:** el sistema responde correctamente dentro del dominio del corpus y reconoce apropiadamente sus propios límites de cobertura, aunque se identificó que el filtro de similitud coseno por sí solo no es suficiente para detectar irrelevancia temática — hallazgo documentado como limitación conocida y área de mejora futura (ej. incorporar un paso de *re-ranking* o un juicio explícito del LLM sobre la relevancia de cada documento recuperado antes de generar la respuesta).

**Trabajo futuro:** ampliar el corpus indexado (actualmente acotado por cuota gratuita) mediante billing habilitado en la API, e incorporar un mecanismo de re-ranking semántico adicional para mejorar la precisión de la detección de "información insuficiente".